In [24]:
#!/usr/bin/env python3

import os
import random
import re
import pandas as pd
import numpy as np
from collections import Counter,defaultdict
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torch import nn
import torch 
import matplotlib.pyplot as plt
import seaborn as sns

datasets = [
    ('category1/AllCombined.txt', 'Wikipedia', True),
    ('category1/paul_graham_essays.txt', 'Paul Graham Essays', True),
    ('category1/warpeace_input.txt', 'War and Peace', True),
    ('category1/sherlock_holmes.txt', 'Sherlock', True),
    ('category2/stacks.txt', 'Stacks', False),
    ('category2/linux_input.txt', 'Linux Code', False),
]

In [2]:
def create_context_target_pairs(file_path, context_length, stoi, max_words=None):
    """
    Optimized function to create context-target pairs from a cleaned sentences file.
    
    Args:
        file_path (str): Path to the text file with cleaned sentences.
        context_length (int): Number of words in the context.
        max_words (int, optional): Maximum number of sentences to process. If None, process all.
    
    Yields:
        tuple: (context_idx, target_idx)
    """
    words_processed = 0
    stoi['<PAD>']=0
    context = [stoi['<PAD>']] * context_length  
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            words = line.strip().split()
            if len(words) <= context_length:
                continue  # Skip lines too short
            if max_words is not None and words_processed >= max_words:
                    break
            for word in words:
                if max_words is not None and words_processed >= max_words:
                    break
                if word == '<start>':
                    context = [stoi['<PAD>']] * context_length  # Pad context when <START> is encountered
                    continue  # Don't yield for <START>
                if word not in stoi:
                    n=len(stoi)
                    stoi[word]=n
                target = stoi[word]
                yield (context, target)
                # Slide the context window
                context = context[1:] + [stoi[word]]
                words_processed += 1

In [3]:
class MLPTextGenerator(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_length, hidden_dims, activation_fn):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        layers = []
        input_dim = embedding_dim * context_length  # context_length is 5
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(input_dim, hidden_dim))
            if activation_fn == 'relu':
                layers.append(nn.ReLU())
            elif activation_fn == 'tanh':
                layers.append(nn.Tanh())
            input_dim = hidden_dim
        layers.append(nn.Linear(input_dim, vocab_size))
        self.mlp = nn.Sequential(*layers)
    def forward(self, x):
        emb = self.embedding(x)
        emb = emb.view(emb.size(0), -1)
        logits = self.mlp(emb)
        return F.log_softmax(logits, dim=-1)
# Model variants
embedding_dims = [32, 64]
hidden_layer_configs = [[1024],[1024, 1024]]
activations = ['relu', 'tanh']


In [4]:
def collect_model_results(folder):
    results = []
    for fname in os.listdir(folder):
        if fname.endswith('_loss.pth'):
            model_name = fname.replace('_loss.pth', '')
            data = torch.load(os.path.join(folder, fname), map_location='cpu', weights_only=False)
            train_losses = data.get('train_losses', [])
            val_losses = data.get('val_losses', [])
            val_accuracies = data.get('val_accuracies', [])
            best_val_acc = max(val_accuracies) if val_accuracies else None
            best_epoch = val_accuracies.index(best_val_acc)+1 if val_accuracies else None
            last_epoch = len(train_losses)
            last_train_loss = train_losses[-1] if train_losses else None
            last_val_loss = val_losses[-1] if val_losses else None
            results.append({
                'Model': model_name,
                'Best Val Accuracy': best_val_acc,
                'Best Epoch': best_epoch,
                'Total Epochs': last_epoch,
                'Last Train Loss': last_train_loss,
                'Last Val Loss': last_val_loss,
                'Folder': os.path.basename(folder)
            })
    return results

struct_results = collect_model_results('./struct_model')
unstruct_results = collect_model_results('./unstruct_model')
all_results = struct_results + unstruct_results
df = pd.DataFrame(all_results)
df_sorted = df.sort_values(by=['Best Val Accuracy'], ascending=True)

# Get best models for each category
unstruct_df = df_sorted[df_sorted["Folder"]=='unstruct_model']
struct_df = df_sorted[df_sorted["Folder"]=='struct_model']
best_unstruct = unstruct_df.iloc[-1]['Model'] if len(unstruct_df) > 0 else None
best_struct = struct_df.iloc[-1]['Model'] if len(struct_df) > 0 else None

print("="*80)
print("Performance of Plain text Models (Unstructured)")
print("-"*80)
print(f"{'Model':<30} {'Best Val Accuracy':<18} {'Total Epochs':<15} {'Last Train Loss':<18} {'Last Val Loss':<15}")
print("-"*80)
# Print each row and highlight the best
for idx, row in unstruct_df.iterrows():
    model_name = row['Model']
    if model_name == best_unstruct:
        print(f"🏆 {model_name:<28} {row['Best Val Accuracy']:.4f}            {int(row['Total Epochs']):<15} {row['Last Train Loss']:.4f}           {row['Last Val Loss']:.4f}")
    else:
        print(f"   {model_name:<28} {row['Best Val Accuracy']:.4f}            {int(row['Total Epochs']):<15} {row['Last Train Loss']:.4f}           {row['Last Val Loss']:.4f}")
print("="*80)
print()
print("Performance of Structured text Models")
print("-"*80)
print(f"{'Model':<30} {'Best Val Accuracy':<18} {'Total Epochs':<15} {'Last Train Loss':<18} {'Last Val Loss':<15}")
print("-"*80)
# Print each row and highlight the best
for idx, row in struct_df.iterrows():
    model_name = row['Model']
    if model_name == best_struct:
        print(f"🏆 {model_name:<28} {row['Best Val Accuracy']:.4f}            {int(row['Total Epochs']):<15} {row['Last Train Loss']:.4f}           {row['Last Val Loss']:.4f}")
    else:
        print(f"   {model_name:<28} {row['Best Val Accuracy']:.4f}            {int(row['Total Epochs']):<15} {row['Last Train Loss']:.4f}           {row['Last Val Loss']:.4f}")
print("="*80)

Performance of Plain text Models (Unstructured)
--------------------------------------------------------------------------------
Model                          Best Val Accuracy  Total Epochs    Last Train Loss    Last Val Loss  
--------------------------------------------------------------------------------
   emb32_hidden1_relu           9.4575            192             5.5985           6.4899
   emb32_hidden2_relu           10.3588            157             5.8935           6.5354
   emb32_hidden1_tanh           10.4325            419             5.1161           6.4717
   emb64_hidden1_relu           10.7500            186             5.4513           6.3861
   emb64_hidden2_relu           11.2750            153             5.7748           6.4328
   emb32_hidden2_tanh           11.3087            300             5.3507           6.3577
   emb64_hidden1_tanh           12.1738            500             4.6361           6.3212
🏆 emb64_hidden2_tanh           12.5887            395

In [5]:
## vocab analysis code
## code to print 10 most frequent words and 10 least frequent words
print("\n" + "="*100)
print("VOCABULARY ANALYSIS OF CATEGORY 1 DATASETS")
print("="*100)
word_counts = defaultdict(int)
for file_path in [f'./unstruct_model/mlp_dataset/{i[1]}_vocab.txt' for i in datasets if i[2]]: ## only category 1 ds
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            word, count = line.strip().split()
            word_counts[word] += int(count)
sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
most_frequent = sorted_words[:10]
least_frequent = sorted_words[-10:]
unstruct_vocab = sorted_words
print("\nMost Frequent Words:")
for word, freq in most_frequent:
    print(f"   {word:<15} : {freq}")
print("\nLeast Frequent Words:")
for word, freq in least_frequent:
    print(f"   {word:<15} : {freq}")


VOCABULARY ANALYSIS OF CATEGORY 1 DATASETS

Most Frequent Words:
   the             : 2117640
   .               : 2102945
   <start>         : 1035574
   of              : 990982
   in              : 966909
   and             : 797614
   a               : 721837
   is              : 583196
   to              : 544943
   was             : 464507

Least Frequent Words:
   awayyou         : 1
   orglicense      : 1
   trademarkcopyright : 1
   pglaf           : 1
   unlink          : 1
   unenforceability : 1
   646221541       : 1
   84116           : 1
   5961887         : 1
   orgcontact      : 1


In [16]:
## plain text model prediction code

context_length = 5
stoi_unstruct = defaultdict(int)
for file_path in [f'./unstruct_model/mlp_dataset/{i[1]}_subset.txt' for i in datasets if i[2]]: ## only category 1 ds
    pairs = list(create_context_target_pairs(file_path, context_length, stoi_unstruct, max_words=100000))  # Limit to 5000 words per dataset for balance
    for context, target in pairs:
        pass
itos_unstruct = {i:w for w,i in stoi_unstruct.items()}

def generate_predictions(model, start_context, itos, stoi, context_length, num_words=5):
    """
    Generate next word predictions given a starting context.
    
    Args:
        model: The trained model
        start_context: List of initial context words
        itos: Index to string mapping
        stoi: String to index mapping
        context_length: Length of context window
        num_words: Number of words to predict
    
    Returns:
        List of predicted words
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()
    
    # Initialize context with padding
    context = [stoi['<PAD>']] * (context_length - len(start_context)) + [stoi[w] for w in start_context if w in stoi]
    predictions = []
    
    with torch.no_grad():
        for _ in range(num_words):
            # Convert context to tensor
            context_tensor = torch.tensor([context], dtype=torch.long).to(device)
            
            # Get prediction
            output = model(context_tensor)
            predicted_idx = output.argmax(dim=1).item()
            predicted_word = itos[predicted_idx]
            predictions.append(predicted_word)
            
            # Update context
            context = context[1:] + [predicted_idx]
    
    return predictions


# Test contexts for prediction
test_contexts = [
    ['first', 'person', 'says', 'that'],
]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("\n" + "="*100)
print("MODEL PREDICTIONS AND PERFORMANCE ANALYSIS")
print("="*100)

# Process unstructured models
print("\n📊 UNSTRUCTURED TEXT MODELS (Structured Predictions)")
print("-"*100)
print(f"{'Model':<35} {'Context':<25} {'Predictions (5 words)':<40} ")
print("-"*100)

for model_name in unstruct_df['Model'].values:
    model_path = f'./unstruct_model/{model_name}.pth'
    if not os.path.exists(model_path):
        print(f"   {model_name:<33} Model not found")
        continue
    
    # Load model checkpoint
    checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)
    
    # Recreate model architecture
    emb_dim = int(model_name.split('_')[0].replace('emb', ''))
    hidden_count = int(model_name.split('_')[1].replace('hidden', ''))
    hidden_dims = [1024] * hidden_count
    act_fn = model_name.split('_')[2]
    
    model = MLPTextGenerator(len(stoi_unstruct), emb_dim, context_length, hidden_dims, act_fn)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Generate predictions for first test context
    context_words = test_contexts[0]
    predictions = generate_predictions(model, context_words, itos_unstruct, stoi_unstruct, context_length, num_words=5)
    pred_str = ' → '.join(predictions)
    
    print(f"   {model_name:<33} {' '.join(context_words):<25} {pred_str:<40}")



MODEL PREDICTIONS AND PERFORMANCE ANALYSIS

📊 UNSTRUCTURED TEXT MODELS (Structured Predictions)
----------------------------------------------------------------------------------------------------
Model                               Context                   Predictions (5 words)                    
----------------------------------------------------------------------------------------------------
   emb32_hidden1_relu                first person says that    i → am → to → be → .                    
   emb32_hidden2_relu                first person says that    it → is → a → lot → of                  
   emb32_hidden1_tanh                first person says that    the → other → . → i → was               
   emb64_hidden1_relu                first person says that    the → same → . → the → same             
   emb64_hidden2_relu                first person says that    he → was → not → to → the               
   emb32_hidden2_tanh                first person says that    the → same → o

In [7]:
## category 2 vocab analysis code
## code to print 10 most frequent words and 10 least frequent words
print("\n" + "="*100)
print("VOCABULARY ANALYSIS OF CATEGORY 2 DATASETS")
print("="*100)
word_counts = defaultdict(int)
for file_path in [f'./struct_model/mlp_dataset/{i[1]}_vocab.txt' for i in datasets if not i[2]]: ## only category 2 ds
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            word, count = line.strip().split()
            word_counts[word] += int(count)
sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
most_frequent_struct = sorted_words[:10]
least_frequent_struct = sorted_words[-10:]
struct_vocab = sorted_words
print("\nMost Frequent Words:")
for word, freq in most_frequent_struct:
    print(f"   {word:<15} : {freq}")
print("\nLeast Frequent Words:")
for word, freq in least_frequent_struct:
    print(f"   {word:<15} : {freq}")


VOCABULARY ANALYSIS OF CATEGORY 2 DATASETS

Most Frequent Words:
   \               : 881498
   $               : 814241
   _               : 557267
   {               : 496648
   }               : 484065
   -               : 383486
   )               : 275860
   (               : 275812
   ,               : 254228
   .               : 229121

Least Frequent Words:
   mangle          : 1
   tack            : 1
   iwgrp           : 1
   iwoth           : 1
   eters           : 1
   gsi             : 1
   gilad           : 1
   yossef          : 1
   occuring        : 1
   entrancy        : 1


In [17]:
## structured text model prediction code
stoi_struct = defaultdict(int)
for file_path in [f'./struct_model/mlp_dataset/{i[1]}_subset.txt' for i in datasets if not i[2]]: ## only category 2 ds
    pairs = list(create_context_target_pairs(file_path, context_length, stoi_struct, max_words=100000))  
    for context, target in pairs:
        pass
itos_struct = {i:w for w,i in stoi_struct.items()}

test_contexts = [
    ['if', '(', 'x', '=','='],
    ['\\', 'begin' ,'{', 'proof'],
]
# Process structured models
print("\n" + "="*100)
print("📊 STRUCTURED TEXT MODELS (Natural Predictions)")
print("-"*100)
print(f"{'Model':<35} {'Context':<20} {'Predictions (5 words)':<40}")
print("-"*100)

for model_name in struct_df['Model'].values:
    model_path = f'./struct_model/{model_name}.pth'
    if not os.path.exists(model_path):
        print(f"   {model_name:<33} Model not found")
        continue
    
    # Load model checkpoint
    checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)
    
    # Recreate model architecture
    emb_dim = int(model_name.split('_')[0].replace('emb', ''))
    hidden_count = int(model_name.split('_')[1].replace('hidden', ''))
    hidden_dims = [1024] * hidden_count
    act_fn = model_name.split('_')[2]
    
    model = MLPTextGenerator(len(stoi_struct), emb_dim, context_length, hidden_dims, act_fn)
    model.load_state_dict(checkpoint['model_state_dict'])
    # Generate predictions for first test context
    context_words = test_contexts[0]
    predictions = generate_predictions(model, context_words, itos_struct, stoi_struct, context_length, num_words=5)
    pred_str = ' '.join(predictions)
    print(f"   {model_name:<33} {' '.join(context_words):<20} {pred_str:<40}")

print("="*100)



📊 STRUCTURED TEXT MODELS (Natural Predictions)
----------------------------------------------------------------------------------------------------
Model                               Context              Predictions (5 words)                   
----------------------------------------------------------------------------------------------------
   emb32_hidden1_relu                if ( x = =           1 ) ) & &                               
   emb32_hidden1_tanh                if ( x = =           1 ) ) | |                               
   emb32_hidden2_relu                if ( x = =           1 ) ) | |                               
   emb64_hidden1_relu                if ( x = =           1 ) ) | |                               
   emb64_hidden2_relu                if ( x = =           0 ) ) ) {                               
   emb64_hidden1_tanh                if ( x = =           0 ) ) | |                               
   emb32_hidden2_tanh                if ( x = =           

In [18]:
# plot tsne of embedding dimensions
from sklearn.manifold import TSNE
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
N=100
embeddings = []
for model_name in unstruct_df['Model'].values:
    model_path = f'./unstruct_model/{model_name}.pth'
    if not os.path.exists(model_path):
        print(f"Model file not found: {model_path}")
        continue
    model = torch.load(model_path, map_location='cpu', weights_only=False)
    embedding = model["model_state_dict"]["embedding.weight"]
    print(embedding.shape)
    filtered_embeddings = embedding[[stoi_unstruct[word[0]] for word in unstruct_vocab[:N]]]
    # create tsne embeddings
    tsne = TSNE(n_components=2, random_state=42, perplexity=30,n_iter_without_progress=1000)
    embedding_2d = tsne.fit_transform(filtered_embeddings)
    embeddings.append((model_name, embedding_2d))


# Plotting text on scatter
print("\n" + "="*100)
print("t-SNE VISUALIZATION OF WORD EMBEDDINGS (Top 100 Most Frequent Words)")
print("="*100)

# Create interactive plotly subplots
import plotly.graph_objects as go

# Create 2x4 subplot grid
fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=[model_name for model_name, _ in embeddings],
    specs=[[{"type": "scatter"} for _ in range(4)] for _ in range(2)]
)

word_labels = [word[0] for word in unstruct_vocab[:N]]


torch.Size([21455, 32])
torch.Size([21455, 32])
torch.Size([21455, 32])
torch.Size([21455, 64])
torch.Size([21455, 64])
torch.Size([21455, 32])
torch.Size([21455, 64])
torch.Size([21455, 64])

t-SNE VISUALIZATION OF WORD EMBEDDINGS (Top 100 Most Frequent Words)


In [ ]:
torch.save(dict(embeddings), 'embeddings_tsne.pth')

In [ ]:
# Interactive plotly visualization of embeddings - 2D and 3D
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
N=1500
print("\n" + "="*100)
print("CREATING INTERACTIVE EMBEDDING VISUALIZATIONS (2D and 3D)")
print("="*100)

# Use best model for visualization
best_model_path = f'./unstruct_model/emb32_hidden2_tanh.pth'
best_model_data = torch.load(best_model_path, map_location='cpu', weights_only=False)
best_model_embedding = best_model_data["model_state_dict"]["embedding.weight"].detach().numpy()

# Get top N words
top_words = [word[0] for word in unstruct_vocab[:N]]
word_indices = [stoi_unstruct[word] for word in top_words]

# Compute t-SNE with 2 components
print("\nComputing 2D t-SNE...")
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=30, n_iter_without_progress=1000)
embeddings_2d = tsne_2d.fit_transform(best_model_embedding[word_indices])


# Create dataframe for plotly
import pandas as pd
df_tsne = pd.DataFrame({
    'x_2d': embeddings_2d[:, 0],
    'y_2d': embeddings_2d[:, 1],
    'word': top_words,
    'frequency': [freq for _, freq in unstruct_vocab[:N]]
})

print(f"\n✓ t-SNE computation complete")
print(f"✓ 2D embeddings shape: {embeddings_2d.shape}")

# Create 2D interactive scatter plot
print("\nCreating 2D visualization...")
fig_2d = go.Figure()

fig_2d.add_trace(go.Scatter(
    x=df_tsne['x_2d'],
    y=df_tsne['y_2d'],
    mode='markers+text',
    text=df_tsne['word'],
    textposition='top center',
    marker=dict(
        size=6,
        color=df_tsne['frequency'],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Frequency"),
        line=dict(width=0.5, color='white')
    ),
    hovertemplate='<b>%{text}</b><br>Frequency: %{customdata}<extra></extra>',
    customdata=df_tsne['frequency']
))

fig_2d.update_layout(
    title=f'2D t-SNE of Word Embeddings<br>({best_unstruct})',
    xaxis_title='t-SNE Dimension 1',
    yaxis_title='t-SNE Dimension 2',
    hovermode='closest',
    width=1000,
    height=800,
    font=dict(size=10)
)

fig_2d.write_html('tsne_embeddings_2d.html')
fig_2d.show()

print("✓ 2D visualization saved as 'tsne_embeddings_2d.html'")


print(f"\n✓ Model: {best_unstruct}")
print(f"✓ Visualized {len(top_words)} most frequent words")


CREATING INTERACTIVE EMBEDDING VISUALIZATIONS (2D and 3D)

Computing 2D t-SNE...

✓ t-SNE computation complete
✓ 2D embeddings shape: (1500, 2)

Creating 2D visualization...


✓ 2D visualization saved as 'tsne_embeddings_2d.html'

✓ Model: emb64_hidden2_tanh
✓ Visualized 1500 most frequent words


In [20]:
# Cluster t-SNE embeddings and visualize
from sklearn.cluster import KMeans
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("\n" + "="*100)
print("CLUSTERING t-SNE EMBEDDINGS INTO 5 GROUPS")
print("="*100)

# Cluster each model's t-SNE embeddings into 5 groups
n_clusters = 5
clustered_embeddings = []
cluster_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']

for model_name, embedding_2d in embeddings:
    kmeans = KMeans(n_clusters=n_clusters, init='k-means++', random_state=42, n_init=10)
    clusters = kmeans.fit_predict(embedding_2d)
    clustered_embeddings.append((model_name, embedding_2d, clusters))
    
    print(f"\n{model_name}:")
    for cluster_id in range(n_clusters):
        cluster_size = np.sum(clusters == cluster_id)
        cluster_words = [word_labels[i] for i in range(len(word_labels)) if clusters[i] == cluster_id]
        print(f"  Cluster {cluster_id}: {cluster_size} words → {', '.join(cluster_words[:8])}")

# Create interactive subplots with clustered t-SNE
print("\n" + "-"*100)
print("Creating clustered t-SNE visualization...")
print("-"*100)

fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=[model_name for model_name, _, _ in clustered_embeddings],
    specs=[[{"type": "scatter"} for _ in range(4)] for _ in range(2)]
)

# Add traces for each model with cluster colors
for i, (model_name, embedding_2d, clusters) in enumerate(clustered_embeddings):
    row = (i // 4) + 1
    col = (i % 4) + 1
    
    # Add scatter plots for each cluster
    for cluster_id in range(n_clusters):
        cluster_mask = clusters == cluster_id
        cluster_2d = embedding_2d[cluster_mask]
        cluster_words = [word_labels[j] for j in range(len(word_labels)) if cluster_mask[j]]
        
        fig.add_trace(
            go.Scatter(
                x=cluster_2d[:, 0],
                y=cluster_2d[:, 1],
                mode='markers+text',
                text=cluster_words,
                textposition='top center',
                marker=dict(
                    size=6,
                    color=cluster_colors[cluster_id],
                    line=dict(width=0.5, color='white'),
                    opacity=0.8
                ),
                textfont=dict(size=7),
                name=f'Cluster {cluster_id}',
                showlegend=(i == 0),  # Only show legend for first subplot
                hovertemplate='<b>%{text}</b><br>Cluster: ' + str(cluster_id) + '<extra></extra>'
            ),
            row=row, col=col
        )

# Update layout
fig.update_layout(
    title='t-SNE Embeddings Clustered into 5 Groups (All Models)',
    height=900,
    width=1800,
    font=dict(size=9),
    showlegend=True,
    hovermode='closest'
)

# Update axes
for i in range(1, 9):
    row = ((i-1) // 4) + 1
    col = ((i-1) % 4) + 1
    fig.update_xaxes(title_text='t-SNE Dim 1', row=row, col=col)
    fig.update_yaxes(title_text='t-SNE Dim 2', row=row, col=col)

fig.write_html('tsne_clustered_comparison.html')
fig.show()

print("\n✓ Clustered visualization saved as 'tsne_clustered_comparison.html'")




CLUSTERING t-SNE EMBEDDINGS INTO 5 GROUPS

emb32_hidden1_relu:
  Cluster 0: 24 words → <start>, and, he, from, not, had, there, her
  Cluster 1: 22 words → of, is, on, by, are, at, an, but
  Cluster 2: 19 words → the, for, they, or, this, has, she, be
  Cluster 3: 18 words → ., a, it, as, his, also, were, have
  Cluster 4: 17 words → in, to, was, that, with, people, after, its

emb32_hidden2_relu:
  Cluster 0: 18 words → is, for, from, are, at, also, not, which
  Cluster 1: 18 words → that, with, his, this, were, but, united, city
  Cluster 2: 22 words → in, and, a, to, as, she, have, had
  Cluster 3: 25 words → <start>, of, he, on, by, an, they, people
  Cluster 4: 17 words → the, ., was, it, or, has, be, one

emb32_hidden1_tanh:
  Cluster 0: 22 words → the, <start>, is, for, by, are, were, be
  Cluster 1: 22 words → and, they, has, have, had, their, other, about
  Cluster 2: 19 words → ., a, was, that, with, at, an, this
  Cluster 3: 17 words → in, to, from, his, born, there, new, b


✓ Clustered visualization saved as 'tsne_clustered_comparison.html'


In [23]:

# Create TSNE comparison plots for all STRUCT models
print("\n" + "="*100)
print("CREATING INTERACTIVE EMBEDDING VISUALIZATIONS FOR STRUCT DATASET (All Models)")
print("="*100)
N=200
# Build stoi_struct and itos_struct from struct vocab
stoi_struct = {word[0]: idx for idx, word in enumerate(struct_vocab)}
itos_struct = {idx: word for word, idx in stoi_struct.items()}

# Get top N words
top_words_struct = [word[0] for word in struct_vocab[:N]]
word_indices_struct = [stoi_struct[word] for word in top_words_struct]

# Compute t-SNE for all struct models
print("\nComputing 2D t-SNE for all struct models...")
embeddings_struct = []
for model_name in struct_df['Model'].values:
    model_path = f'./struct_model/{model_name}.pth'
    if not os.path.exists(model_path):
        print(f"Model file not found: {model_path}")
        continue
    model = torch.load(model_path, map_location='cpu', weights_only=False)
    embedding = model["model_state_dict"]["embedding.weight"]
    filtered_embeddings = embedding[[stoi_struct[word] for word in top_words_struct]]
    # create tsne embeddings
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter_without_progress=1000)
    embedding_2d = tsne.fit_transform(filtered_embeddings.detach().numpy())
    embeddings_struct.append((model_name, embedding_2d))

print(f"✓ Computed t-SNE for {len(embeddings_struct)} struct models")

# Create interactive subplots with t-SNE for all struct models
print("\nCreating t-SNE comparison visualization for struct...")
print("-"*100)

fig_struct_comparison = make_subplots(
    rows=2, cols=4,
    subplot_titles=[model_name for model_name, _ in embeddings_struct],
    specs=[[{"type": "scatter"} for _ in range(4)] for _ in range(2)]
)

# Add traces for each struct model
for i, (model_name, embedding_2d) in enumerate(embeddings_struct):
    row = (i // 4) + 1
    col = (i % 4) + 1
    
    fig_struct_comparison.add_trace(
        go.Scatter(
            x=embedding_2d[:, 0],
            y=embedding_2d[:, 1],
            mode='markers+text',
            text=top_words_struct,
            textposition='top center',
            marker=dict(
                size=5,
                color=[freq for _, freq in struct_vocab[:N]],
                colorscale='Viridis',
                showscale=(i == 0),  # Show colorbar only for first subplot
                colorbar=dict(title="Frequency", x=1.02),
                line=dict(width=0.5, color='white'),
                opacity=0.7
            ),
            textfont=dict(size=6),
            hovertemplate='<b>%{text}</b><extra></extra>',
            showlegend=False
        ),
        row=row, col=col
    )

# Update layout
fig_struct_comparison.update_layout(
    title='t-SNE Embeddings Comparison (All Struct Models)',
    height=900,
    width=1800,
    font=dict(size=9),
    hovermode='closest'
)

# Update axes
for i in range(1, 9):
    row = ((i-1) // 4) + 1
    col = ((i-1) % 4) + 1
    fig_struct_comparison.update_xaxes(title_text='t-SNE Dim 1', row=row, col=col)
    fig_struct_comparison.update_yaxes(title_text='t-SNE Dim 2', row=row, col=col)

fig_struct_comparison.write_html('tsne_embeddings_struct_comparison.html')
fig_struct_comparison.show()

print("✓ t-SNE comparison saved as 'tsne_embeddings_struct_comparison.html'")

print("\n" + "="*100)
print("CLUSTERING t-SNE EMBEDDINGS (STRUCT) INTO 5 GROUPS")
print("="*100)

# Cluster each struct model's t-SNE embeddings into 5 groups
n_clusters = 5
clustered_embeddings_struct = []

for model_name, embedding_2d in embeddings_struct:
    kmeans = KMeans(n_clusters=n_clusters, init='k-means++', random_state=42, n_init=10)
    clusters = kmeans.fit_predict(embedding_2d)
    clustered_embeddings_struct.append((model_name, embedding_2d, clusters))
    
    print(f"\n{model_name}:")
    for cluster_id in range(n_clusters):
        cluster_size = np.sum(clusters == cluster_id)
        cluster_words = [top_words_struct[i] for i in range(len(top_words_struct)) if clusters[i] == cluster_id]
        print(f"  Cluster {cluster_id}: {cluster_size} words → {', '.join(cluster_words[:8])}")

# Create interactive subplots with clustered t-SNE
print("\n" + "-"*100)
print("Creating clustered t-SNE visualization for struct models...")
print("-"*100)

fig_struct_clustered = make_subplots(
    rows=2, cols=4,
    subplot_titles=[model_name for model_name, _, _ in clustered_embeddings_struct],
    specs=[[{"type": "scatter"} for _ in range(4)] for _ in range(2)]
)

# Add traces for each model with cluster colors
for i, (model_name, embedding_2d, clusters) in enumerate(clustered_embeddings_struct):
    row = (i // 4) + 1
    col = (i % 4) + 1
    
    # Add scatter plots for each cluster
    for cluster_id in range(n_clusters):
        cluster_mask = clusters == cluster_id
        cluster_2d = embedding_2d[cluster_mask]
        cluster_words = [top_words_struct[j] for j in range(len(top_words_struct)) if cluster_mask[j]]
        
        fig_struct_clustered.add_trace(
            go.Scatter(
                x=cluster_2d[:, 0],
                y=cluster_2d[:, 1],
                mode='markers+text',
                text=cluster_words,
                textposition='top center',
                marker=dict(
                    size=6,
                    color=cluster_colors[cluster_id],
                    line=dict(width=0.5, color='white'),
                    opacity=0.8
                ),
                textfont=dict(size=7),
                name=f'Cluster {cluster_id}',
                showlegend=(i == 0),  # Only show legend for first subplot
                hovertemplate='<b>%{text}</b><br>Cluster: ' + str(cluster_id) + '<extra></extra>'
            ),
            row=row, col=col
        )

# Update layout
fig_struct_clustered.update_layout(
    title='t-SNE Embeddings Clustered into 5 Groups (All Struct Models)',
    height=900,
    width=1800,
    font=dict(size=9),
    showlegend=True,
    hovermode='closest'
)

# Update axes
for i in range(1, 9):
    row = ((i-1) // 4) + 1
    col = ((i-1) % 4) + 1
    fig_struct_clustered.update_xaxes(title_text='t-SNE Dim 1', row=row, col=col)
    fig_struct_clustered.update_yaxes(title_text='t-SNE Dim 2', row=row, col=col)

fig_struct_clustered.write_html('tsne_clustered_struct_comparison.html')
fig_struct_clustered.show()

print("\n✓ Clustered comparison saved as 'tsne_clustered_struct_comparison.html'")



CREATING INTERACTIVE EMBEDDING VISUALIZATIONS FOR STRUCT DATASET (All Models)

Computing 2D t-SNE for all struct models...
✓ Computed t-SNE for 8 struct models

Creating t-SNE comparison visualization for struct...
----------------------------------------------------------------------------------------------------


✓ t-SNE comparison saved as 'tsne_embeddings_struct_comparison.html'

CLUSTERING t-SNE EMBEDDINGS (STRUCT) INTO 5 GROUPS

emb32_hidden1_relu:
  Cluster 0: 38 words → }, a, ', *, ;, s, /, m
  Cluster 1: 33 words → \, _, {, -, (, ., lemma, in
  Cluster 2: 39 words → $, mathcal, to, is, i, f, r, that
  Cluster 3: 46 words → ,, the, x, of, =, 1, we, y
  Cluster 4: 44 words → ), ^, n, k, ], 0, let, end

emb32_hidden1_tanh:
  Cluster 0: 43 words → \, ), ., the, x, mathcal, ', ^
  Cluster 1: 36 words → }, to, is, of, f, and, 1, ref
  Cluster 2: 47 words → $, lemma, s, u, ], if, &, t
  Cluster 3: 40 words → _, {, -, ,, =, in, we, n
  Cluster 4: 34 words → (, a, i, *, r, [, begin, :

emb32_hidden2_relu:
  Cluster 0: 41 words → {, a, the, mathcal, is, f, and, s
  Cluster 1: 44 words → }, -, to, lemma, ^, =, i, *
  Cluster 2: 45 words → _, ), ,, of, u, we, /, ]
  Cluster 3: 32 words → \, $, ., x, ', 1, k, d
  Cluster 4: 38 words → (, ;, in, r, ref, end, &, g

emb64_hidden1_relu:
  Cluster 0: 48 w


✓ Clustered comparison saved as 'tsne_clustered_struct_comparison.html'
